In [ ]:
#TODO Find what that NAN value is in part 2 and see if it can be fixed. If not, just put the median for that value or throw it away since its one data point
import mat73 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path #Had some issues with scope so I just imported Path to make sure it works no matter what.

In [ ]:
def check_part(part_number):
    # Builds the file name depending on which part we are checking
    file_path = Path(f"Part_{part_number}.mat")
    key = f"Part_{part_number}"

    print("\n" + "=" * 60)
    print(f"CHECKING PART {part_number}")
    print("=" * 60)

   
    if not file_path.exists():
        print(f"Part_{part_number}.mat was NOT found.")
        return

    print(f"Loading Part_{part_number}...")

    data = mat73.loadmat(file_path)

    print("Loaded!")
    print("Keys:", data.keys())

    # Makes sure the part we expect is actually inside the file
    if key not in data:
        print(f"Could not find '{key}' inside the file.")
        return

    records = data[key]

    print("Number of records:", len(records))



    shapes = set()
    shape_counts = {}

    record_lengths = []

    empty_records = 0
    nan_records = 0
    infinite_records = 0
    unexpected_shapes = 0


    # The data is sampled 125 times per second
    FS = 125

    # 5 seconds of data = 625 samples
    window_size = FS * 5

   
    ppg_stds = []

    nan_windows = 0


    # Go through every record in this part so we can analyze if theres any quality issues in the data
    for record_index, record in enumerate(records):

        record = np.asarray(record)

       
        shapes.add(record.shape)

        if record.shape not in shape_counts:
            shape_counts[record.shape] = 0

        shape_counts[record.shape] += 1


        # Check if the record somehow has no data (Side note: Part 2 has 1 NAN value that I need to look at)
        if record.size == 0:
            empty_records += 1
            continue


        # Each record should have 3 rows:
        # PPG, ABP, and ECG
        if record.ndim != 2 or record.shape[0] != 3:
            unexpected_shapes += 1
            continue


        # Save how long this recording is
        record_lengths.append(record.shape[1])


        #Quality Check (Part 2 has a NAN):
        if np.isnan(record).any():
            nan_records += 1

        if np.isinf(record).any():
            infinite_records += 1


        # NOTE: PPG is the first row
        ppg = record[0]


        # Break the PPG signal into 5 second sections (FROM THE NATURE BENCHMARK)
        for start in range(0, len(ppg), window_size):

            end = start + window_size

            # If there isn't enough data left for a full 5 seconds,
            # just ignore that leftover piece
            if end > len(ppg):
                break

            ppg_window = ppg[start:end]


            # Skip the window if it has missing data
            if np.isnan(ppg_window).any():
                nan_windows += 1
                continue


            # Skip it if it somehow has infinity in it too
            if np.isinf(ppg_window).any():
                continue


            # Standard deviation tells us how much the PPG moves around
            ppg_std = np.std(ppg_window)

            ppg_stds.append(ppg_std)


    # Show what we found from the basic data checks
    print("\n--- DATA QUALITY RESULTS ---")

    print("Empty records:", empty_records)
    print("Records containing NaN:", nan_records)
    print("Records containing infinity:", infinite_records)
    print("Records with unexpected shapes:", unexpected_shapes)


    # Show some information about how long the recordings are
    if len(record_lengths) > 0:

        record_lengths = np.array(record_lengths)

        print("\n--- RECORD LENGTH STATISTICS ---")

        print("Minimum:", record_lengths.min())
        print("Maximum:", record_lengths.max())
        print("Mean:", record_lengths.mean())
        print("Median:", np.median(record_lengths))
        print("Standard deviation:", record_lengths.std())


        print("\n--- RECORDING TIME ---")

        print(
            "Shortest recording:",
            record_lengths.min() / FS,
            "seconds"
        )

        print(
            "Longest recording:",
            record_lengths.max() / FS,
            "seconds"
        )


    # Quick summary of whether anything obvious was wrong
    print("\n--- BASIC CLEANLINESS SUMMARY ---")

    if (
        empty_records == 0
        and nan_records == 0
        and infinite_records == 0
        and unexpected_shapes == 0
    ):

        print(
            f"Part {part_number} passed the basic structural "
            "data-quality checks."
        )

    else:

        print(
            f"Part {part_number} has some issues that "
            "need to be investigated."
        )


    # Turn the PPG standard deviations into an array
    # so we can get statistics from them
    ppg_stds = np.array(ppg_stds)

    print("\n--- PPG 5-SECOND WINDOW STANDARD DEVIATIONS ---")

    print("Number of usable windows:", len(ppg_stds))
    print("Windows skipped because of NaN:", nan_windows)

    if len(ppg_stds) > 0:

        print("Minimum PPG STD:", ppg_stds.min())
        print("Maximum PPG STD:", ppg_stds.max())
        print("Mean PPG STD:", ppg_stds.mean())
        print("Median PPG STD:", np.median(ppg_stds))


        # Make a histogram so we can see how the PPG variation
        # is spread out across the whole dataset part
        plt.figure(figsize=(10, 6))

        plt.hist(
            ppg_stds,
            bins=50
        )

        plt.title(
            f"Part {part_number} - Frequency of PPG Standard Deviations"
        )

        plt.xlabel("PPG Standard Deviation")
        plt.ylabel("Frequency")

        plt.grid(axis="y", alpha=0.3)

        plt.show()

In [ ]:
check_part(1)
check_part(2)
check_part(3)
check_part(4)

In [ ]:
# The dataset is sampled 125 times per second
FS = 125

In [ ]:
#Looking closer at the small cluster of data that all 4 datasets have
def show_ppg_waveforms(part_number, number_to_show=5):

    file_path = Path(f"Part_{part_number}.mat")
    key = f"Part_{part_number}"

    print(f"Loading Part {part_number}...")

    data = mat73.loadmat(file_path)
    records = data[key]

    FS = 125
    window_size = FS * 5

    # These are the two groups we saw in the histograms
    low_group = []
    normal_group = []


    # Go through every record and find windows that fit each group
    for record_index, record in enumerate(records):

        record = np.asarray(record)

        # Skip anything with a weird shape
        if record.ndim != 2 or record.shape[0] != 3:
            continue

        ppg = record[0]

        for start in range(0, len(ppg), window_size):

            end = start + window_size

            # Ignore leftover pieces shorter than 5 seconds
            if end > len(ppg):
                break

            ppg_window = ppg[start:end]

            # Skip bad values
            if np.isnan(ppg_window).any() or np.isinf(ppg_window).any():
                continue

            ppg_std = np.std(ppg_window)


            # The smaller group from the histogram
            if 0.10 <= ppg_std <= 0.20:
                low_group.append(
                    (record_index, start, ppg_std, ppg_window.copy())
                )


            # The main group from the histogram
            if 0.55 <= ppg_std <= 0.65:
                normal_group.append(
                    (record_index, start, ppg_std, ppg_window.copy())
                )


    print("Low STD windows found:", len(low_group))
    print("Normal STD windows found:", len(normal_group))


    # Pick random examples instead of only using the first few
    rng = np.random.default_rng(42)

    low_choices = rng.choice(
        len(low_group),
        size=min(number_to_show, len(low_group)),
        replace=False
    )

    normal_choices = rng.choice(
        len(normal_group),
        size=min(number_to_show, len(normal_group)),
        replace=False
    )


    # 5 seconds worth of time values for the x-axis
    time = np.arange(window_size) / FS


    # Showing the low standard deviation waveforms
    print("\nLOW STD PPG WINDOWS")

    for choice in low_choices:

        record_index, start, ppg_std, ppg_window = low_group[choice]

        plt.figure(figsize=(10, 4))

        plt.plot(time, ppg_window)

        plt.title(
            f"Part {part_number} - Low STD PPG\n"
            f"Record {record_index}, STD = {ppg_std:.3f}"
        )

        plt.xlabel("Time (seconds)")
        plt.ylabel("PPG Amplitude")
        plt.grid()

        plt.show()


    # Show the main-group waveforms
    print("\nNORMAL STD PPG WINDOWS")

    for choice in normal_choices:

        record_index, start, ppg_std, ppg_window = normal_group[choice]

        plt.figure(figsize=(10, 4))

        plt.plot(time, ppg_window)

        plt.title(
            f"Part {part_number} - Normal STD PPG\n"
            f"Record {record_index}, STD = {ppg_std:.3f}"
        )

        plt.xlabel("Time (seconds)")
        plt.ylabel("PPG Amplitude")
        plt.grid()

        plt.show()

In [ ]:
show_ppg_waveforms(1)
show_ppg_waveforms(2)
show_ppg_waveforms(3)
show_ppg_waveforms(4)